# ICU Bed Demand Forecasting
## Using COVID-19 Active Cases as an ICU Demand Proxy

**Notebook #13 of 50 — Time Series Forecasting Portfolio**

| | |
|---|---|
| **Dataset** | COVID-19 Dataset — 100 Countries (`saurabhshahane/covid19-dataset-with-100-world-countries-data`) |
| **Target** | Daily active COVID-19 cases (proxy for ICU demand) |
| **Frequency** | Daily (`D`) |
| **Primary TS Library** | Darts (ExponentialSmoothing + N-BEATS style experiments) |
| **Key note** | This notebook uses active-case counts as a **proxy** for ICU need. Real ICU demand = active cases × hospitalisation rate × ICU escalation rate. |

## Learning Objectives
1. Work with COVID-19 panel data and extract a single-country daily series
2. Understand the challenges of forecasting during epidemic growth and decay phases
3. Apply Darts for unified classical + neural experiments
4. Discuss why epidemic series break standard stationarity assumptions
5. Handle structural breaks and regime changes in time series

## Problem Statement
During a public health emergency, ICU capacity planning requires short-term forecasts
of how many critically-ill patients may need intensive care.

**Proxy approach**: We forecast daily active COVID-19 cases and acknowledge that
actual ICU need is a downstream function:
```
ICU demand ≈ active_cases × hospitalisation_rate × icu_escalation_rate
```
For this educational notebook, we focus on the forecasting methodology.

> Goal: Forecast the next 14 days of active cases for a selected country.

## Dataset Overview
**Source:** Kaggle — `saurabhshahane/covid19-dataset-with-100-world-countries-data`

**License:** Check Kaggle dataset page.

**Expected columns**: `Date`, `Country/Region`, `Confirmed`, `Deaths`, `Recovered`, `Active`

**Country selection**: We pick the USA (largest case count) as primary series.
The notebook parameterises the country so you can easily switch.

## Environment Setup

In [ ]:
import subprocess, sys
for imp, pkg in {"kagglehub":"kagglehub","darts":"darts",
                  "statsforecast":"statsforecast", "mlforecast":"mlforecast",
                  "lightgbm":"lightgbm", "lazypredict":"lazypredict",
                  "flaml":"flaml[automl]"}.items():
    try: __import__(imp)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Packages ready.")

## Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive

from darts import TimeSeries
from darts.models import ExponentialSmoothing, AutoARIMA as DartsAutoARIMA, NaiveSeasonal
from darts.metrics import mae as darts_mae, rmse as darts_rmse, mape as darts_mape

pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True})
sns.set_theme(style="whitegrid")

def metrics(actual, pred, label):
    a=np.asarray(actual).ravel(); p=np.asarray(pred).ravel()
    n=min(len(a),len(p)); a,p=a[:n],p[:n]
    mae=mean_absolute_error(a,p); rmse=np.sqrt(mean_squared_error(a,p))
    mask=a!=0
    mape=np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100 if mask.sum()>0 else float("nan")
    print(f"{label:<40s}  MAE={mae:>12,.0f}  RMSE={rmse:>12,.0f}  MAPE={mape:>7.2f}%")
    return {"Model":label,"MAE":round(mae,0),"RMSE":round(rmse,0),"MAPE":round(mape,2)}
print("Imports OK.")

## Configuration

In [ ]:
KAGGLE_SLUG  = "saurabhshahane/covid19-dataset-with-100-world-countries-data"
COUNTRY      = "US"    # Change to any country in the dataset
DATE_COL     = "Date"
TARGET_COL   = "Active"
FREQ         = "D"
SEASON_LEN   = 7
HORIZON      = 14
TEST_DAYS    = 14
VAL_DAYS     = 14
FLAML_BUDGET = 90
RANDOM_STATE = 42
print(f"Country: {COUNTRY}")

## Kaggle Credential Check

In [ ]:
import os, pathlib as _pl
_ok = False
if os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN"):
    print("[OK] Kaggle env vars found."); _ok = True
_j = _pl.Path.home() / ".kaggle" / "kaggle.json"
if _j.exists():
    print(f"[OK] kaggle.json at {_j}"); _ok = True
if not _ok:
    print("WARNING: Kaggle credentials missing.")
    print("  Option A: set KAGGLE_USERNAME + KAGGLE_KEY env vars")
    print("  Option B: place kaggle.json at ~/.kaggle/kaggle.json")
    raise EnvironmentError("Set Kaggle credentials and restart.")

## Dataset Download & Loading

In [ ]:
import kagglehub
data_path=pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
csv_files=sorted(data_path.rglob("*.csv"))
print([f.name for f in csv_files])
# Load the main file (usually covid_19_data.csv or similar)
df=pd.read_csv(csv_files[0])
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

## Data Validation & Country Extraction

In [ ]:
# Detect country column
country_col = next((c for c in df.columns
                    if any(kw in c.lower() for kw in ["country","region","location"])), None)
print(f"Country column: {country_col}")
print(f"Available countries (top 20): {sorted(df[country_col].unique())[:20]}" if country_col else "No country column found")

# Try to match COUNTRY
available = df[country_col].unique() if country_col else []
if COUNTRY not in available:
    best_match = [c for c in available if COUNTRY.lower() in c.lower()]
    COUNTRY_USE = best_match[0] if best_match else available[0]
    print(f"'{COUNTRY}' not exact match; using '{COUNTRY_USE}'")
else:
    COUNTRY_USE = COUNTRY
print(f"Using: {COUNTRY_USE}")

In [ ]:
# Parse date
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce", infer_datetime_format=True)
df = df.dropna(subset=[DATE_COL])

# Filter to selected country
df_country = df[df[country_col]==COUNTRY_USE].copy() if country_col else df.copy()
df_country = df_country.sort_values(DATE_COL).reset_index(drop=True)
print(f"Rows for {COUNTRY_USE}: {len(df_country)}")
print(f"Date range: {df_country[DATE_COL].min().date()} → {df_country[DATE_COL].max().date()}")

# Detect target column
target_actual = None
for c in df_country.columns:
    if TARGET_COL.lower() in c.lower() and df_country[c].dtype in [np.float64, np.int64]:
        target_actual=c; break
if target_actual is None:
    num_cols=df_country.select_dtypes(include=[np.number]).columns.tolist()
    target_actual=num_cols[0] if num_cols else None
print(f"Target column: {target_actual}")

In [ ]:
# Build daily series
daily_raw = df_country.groupby(df_country[DATE_COL].dt.date)[target_actual].max().reset_index()
daily_raw.columns=["ds","y"]
daily_raw["ds"]=pd.to_datetime(daily_raw["ds"])
daily_raw=daily_raw.sort_values("ds").reset_index(drop=True)
full_idx=pd.date_range(daily_raw["ds"].min(),daily_raw["ds"].max(),freq="D")
daily=daily_raw.set_index("ds").reindex(full_idx).reset_index()
daily.columns=["ds","y"]
daily["y"]=daily["y"].interpolate("linear").clip(lower=0)
print(f"Series: {len(daily)} days")
print(daily["y"].describe().apply(lambda x: f"{x:,.0f}"))

## EDA

In [ ]:
fig=px.line(daily,x="ds",y="y",
            title=f"COVID-19 Active Cases — {COUNTRY_USE} (ICU Demand Proxy)",
            labels={"ds":"Date","y":"Active Cases"})
fig.update_layout(template="plotly_white")
fig.show()
print("Note: This series is non-stationary with multiple epidemic waves.")
print("Each wave represents a structural regime change — a major forecasting challenge.")

In [ ]:
# First difference to check stationarity of changes
daily["diff1"] = daily["y"].diff()
adf=adfuller(daily["y"].dropna())
adf_d=adfuller(daily["diff1"].dropna())
print(f"ADF levels  : stat={adf[0]:.3f}  p={adf[1]:.4f}  → {'Stationary' if adf[1]<0.05 else 'NON-STATIONARY'}")
print(f"ADF diff(1) : stat={adf_d[0]:.3f}  p={adf_d[1]:.4f}  → {'Stationary' if adf_d[1]<0.05 else 'NON-STATIONARY'}")
fig,axes=plt.subplots(1,2,figsize=(14,4))
plot_acf(daily["y"].dropna(), lags=min(60,len(daily)//3), ax=axes[0], title="ACF — Active Cases")
plot_pacf(daily["diff1"].dropna(), lags=min(60,len(daily)//3), ax=axes[1], title="PACF — First Difference")
plt.tight_layout(); plt.show()

## Structural Break Discussion
COVID-19 data has multiple **structural breaks** (wave transitions) that violate
the stationarity assumption of most statistical models.

**Implications for forecasting**:
- Models trained before a wave peak will under-predict during the peak
- Models trained at the peak will over-predict during the decline
- **Short training windows** (last 90 days) often outperform full-history training
- **AutoARIMA** with differencing handles level changes but not direction reversals

This is a fundamental limitation — not a model flaw.

## Target Analysis

In [ ]:
y=daily["y"]
print(f"Mean={y.mean():,.0f}  Median={y.median():,.0f}")
print(f"Std={y.std():,.0f}  CV={y.std()/y.mean():.3f}")
print(f"Max single day: {y.max():,.0f}")
fig,axes=plt.subplots(1,2,figsize=(14,4))
axes[0].hist(y,bins=40,color="steelblue",edgecolor="white"); axes[0].set_title("Distribution")
pd.plotting.lag_plot(y,lag=7,ax=axes[1],alpha=0.3); axes[1].set_title("Lag-7 Plot")
plt.tight_layout(); plt.show()

## Train / Validation / Test Split

In [ ]:
n=len(daily); test_start=n-TEST_DAYS; val_start=test_start-VAL_DAYS
ts_train=daily.iloc[:val_start].copy(); ts_val=daily.iloc[val_start:test_start].copy()
ts_test=daily.iloc[test_start:].copy(); ts_trainval=daily.iloc[:test_start].copy()
print(f"Train={len(ts_train)} | Val={len(ts_val)} | Test={len(ts_test)}")
assert ts_train["ds"].max()<ts_val["ds"].min()
assert ts_val["ds"].max()<ts_test["ds"].min()
fig=go.Figure()
for s,c,n_ in [(ts_train,"royalblue","Train"),(ts_val,"orange","Val"),(ts_test,"crimson","Test")]:
    fig.add_trace(go.Scatter(x=s["ds"],y=s["y"],name=n_,line=dict(color=c,width=1.5)))
fig.update_layout(title="Temporal Split — COVID Active Cases",template="plotly_white")
fig.show()

## Feature Engineering

In [ ]:
def build_feats(df_in):
    df=df_in.copy().sort_values("ds").reset_index(drop=True); y=df["y"]
    for lag in [1,2,3,7,14]: df[f"lag_{lag}"]=y.shift(lag)
    for w in [7,14,28]:
        df[f"rmean_{w}"]=y.shift(1).rolling(w).mean()
        df[f"rstd_{w}"]=y.shift(1).rolling(w).std()
    df["growth_7"]=(y.shift(1)/y.shift(8).replace(0,np.nan)-1)  # 7-day growth rate
    df["dow"]=df["ds"].dt.dayofweek; df["month"]=df["ds"].dt.month
    return df
full_feat=build_feats(daily)
feat_train=full_feat.iloc[:val_start].dropna().copy()
feat_val=full_feat.iloc[val_start:test_start].dropna().copy()
feat_test=full_feat.iloc[test_start:].dropna().copy()
feat_trainval=full_feat.iloc[:test_start].dropna().copy()
FEAT_COLS=[c for c in feat_train.columns if c not in ["ds","y"]]
print(f"Features ({len(FEAT_COLS)}): {FEAT_COLS}")

## Baselines

In [ ]:
results=[]; y_test=ts_test["y"].values; last_tv=ts_trainval["y"].values
results.append(metrics(y_test,np.full(TEST_DAYS,last_tv[-1]),"Naive"))
sn=np.tile(last_tv[-SEASON_LEN:],TEST_DAYS//SEASON_LEN+1)[:TEST_DAYS]
results.append(metrics(y_test,sn,"Seasonal Naive (7-day)"))
results.append(metrics(y_test,np.full(TEST_DAYS,last_tv[-7:].mean()),"MA-7"))

## LazyPredict

In [ ]:
try:
    lz=LazyRegressor(verbose=0,ignore_warnings=True)
    lm,_=lz.fit(feat_train[FEAT_COLS],feat_val[FEAT_COLS],feat_train["y"],feat_val["y"])
    print(lm.sort_values("RMSE").head(10).to_string())
except Exception as e: print(f"LazyPredict: {e}"); lm=None

## FLAML AutoML

In [ ]:
flaml=AutoML()
flaml.fit(feat_trainval[FEAT_COLS],feat_trainval["y"],task="regression",metric="rmse",
          time_budget=FLAML_BUDGET,verbose=0,seed=RANDOM_STATE)
print(f"Best: {flaml.best_estimator}")
flaml_pred=flaml.predict(feat_test[FEAT_COLS]) if len(feat_test)>0 else None
if flaml_pred is not None:
    results.append(metrics(feat_test["y"].values,flaml_pred,f"FLAML ({flaml.best_estimator})"))

## Darts — ExponentialSmoothing & AutoARIMA

**Why Darts for this dataset?**

Darts provides a unified API for both classical (ETS, ARIMA) and neural models.
For epidemic series, `ExponentialSmoothing` (with additive trend + no seasonality)
often outperforms complex seasonal models because regime changes override any
stable seasonal pattern.

`AutoARIMA` automatically selects the best differencing order and AR/MA terms,
critical for a series with multiple epidemic waves.

In [ ]:
# Convert to Darts TimeSeries
train_darts = TimeSeries.from_dataframe(ts_trainval, time_col="ds", value_cols="y", freq=FREQ)
test_darts  = TimeSeries.from_dataframe(ts_test,     time_col="ds", value_cols="y", freq=FREQ)

darts_preds = {}

# ExponentialSmoothing (additive trend, no seasonality — epidemic waves aren't seasonal)
try:
    ets_d = ExponentialSmoothing(trend="additive", seasonal=None)
    ets_d.fit(train_darts)
    ets_fc = ets_d.predict(HORIZON)
    darts_preds["Darts-ETS"] = ets_fc.values().flatten()[:TEST_DAYS]
    results.append(metrics(y_test, darts_preds["Darts-ETS"], "Darts-ETS"))
except Exception as e: print(f"Darts ETS: {e}")

# AutoARIMA via Darts
try:
    arima_d = DartsAutoARIMA()
    arima_d.fit(train_darts)
    arima_fc = arima_d.predict(HORIZON)
    darts_preds["Darts-AutoARIMA"] = arima_fc.values().flatten()[:TEST_DAYS]
    results.append(metrics(y_test, darts_preds["Darts-AutoARIMA"], "Darts-AutoARIMA"))
except Exception as e: print(f"Darts AutoARIMA: {e}")

# Naive Seasonal (weekly baseline)
try:
    sn_d = NaiveSeasonal(K=SEASON_LEN)
    sn_d.fit(train_darts)
    sn_fc = sn_d.predict(HORIZON)
    darts_preds["Darts-SeasonalNaive"] = sn_fc.values().flatten()[:TEST_DAYS]
    results.append(metrics(y_test, darts_preds["Darts-SeasonalNaive"], "Darts-SeasonalNaive"))
except Exception as e: print(f"Darts SeasonalNaive: {e}")

## Top 3 & Plots

In [ ]:
res_df=pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print(res_df.to_string()); top3=res_df.head(3)
print("\n>>> TOP 3 <<<"); print(top3.to_string(index=False))
fig=px.bar(res_df,x="Model",y="RMSE",color="RMSE",color_continuous_scale="RdYlGn_r",
           title="ICU Demand Proxy — Model Comparison")
fig.update_layout(template="plotly_white",xaxis_tickangle=-35); fig.show()

In [ ]:
fig=go.Figure()
fig.add_trace(go.Scatter(x=ts_trainval["ds"].tail(90),y=ts_trainval["y"].tail(90),
    name="Train (90-day context)",line=dict(color="royalblue",dash="dot",width=1)))
fig.add_trace(go.Scatter(x=ts_test["ds"],y=ts_test["y"],
    name="Actual",line=dict(color="black",width=2)))
all_preds={}
if flaml_pred is not None: all_preds[f"FLAML ({flaml.best_estimator})"]=flaml_pred
all_preds.update(darts_preds)
colors=["#e15759","#f28e2b","#59a14f"]
for i,(_,row) in enumerate(top3.iterrows()):
    m=row["Model"]
    if m in all_preds:
        fig.add_trace(go.Scatter(x=ts_test["ds"][:len(all_preds[m])],y=all_preds[m],
            name=f"#{i+1} {m}",line=dict(color=colors[i],width=2)))
fig.update_layout(title=f"ICU Demand Proxy Forecast — {COUNTRY_USE}",
                  template="plotly_white",xaxis_title="Date",yaxis_title="Active Cases")
fig.show()

## Error Analysis

In [ ]:
best_m=top3.iloc[0]["Model"]
if best_m in all_preds:
    bp=np.asarray(all_preds[best_m]).ravel(); ba=y_test[:len(bp)]; err=ba-bp
    print(f"Bias={err.mean():+,.0f}  Std={err.std():,.0f}")
    fig,ax=plt.subplots(1,2,figsize=(14,4))
    ax[0].hist(err,bins=15,color="steelblue",edgecolor="white")
    ax[0].axvline(0,color="red",linestyle="--"); ax[0].set_title("Error Distribution")
    ax[1].scatter(ba,bp,s=50,alpha=0.8,color="steelblue")
    lim=max(ba.max(),bp.max())*1.1; ax[1].plot([0,lim],[0,lim],"r--")
    ax[1].set_xlabel("Actual"); ax[1].set_ylabel("Predicted"); ax[1].set_title("Actual vs Predicted")
    plt.tight_layout(); plt.show()

## Important Limitations and Caveats

> **This notebook uses active cases as a proxy — not a direct ICU demand signal.**

Real ICU demand requires:
1. **Hospitalisation rate** (varies by variant, vaccination status, age)
2. **ICU escalation rate** (% of hospitalised → ICU)
3. **Length of stay** (occupancy = admissions × LOS)
4. **Reporting lags** — active case counts have 5–14 day confirmation delays

For actual ICU planning, use hospitalization data (CDC, HHS) not case counts.

### Other limitations
1. Structural breaks between epidemic waves violate stationarity — all models struggle
2. Short test window may happen to fall in a stable phase, giving unrealistically good MAPE
3. Country-level aggregation hides regional hospital capacity constraints

## How to Improve
1. Use HHS hospitalization data (direct ICU admissions) instead of case counts
2. Add vaccination coverage as a covariate (reduces ICU escalation rate)
3. Use a rolling 14-day window retrain (epidemic series are non-stationary; stale models fail)
4. Model each epidemic wave separately and ensemble forecasts
5. Conformal prediction intervals are critical here — uncertainty quantification matters more than point accuracy

## Summary
- Used COVID-19 active cases as a proxy for ICU demand
- Explored non-stationary epidemic series with multiple regime changes
- Applied Darts (ETS + AutoARIMA), FLAML, and baselines
- Discussed the critical limitations of case-count proxies for actual ICU planning
- Key lesson: forecasting methodology is sound; the data quality and proxy choice
  are the binding constraints for real healthcare applications

---
*Notebook #13 of 50 — Time Series Forecasting Portfolio*
**Disclaimer**: For educational purposes only. Not suitable for actual clinical decision-making.